# Full ERA5 atmospheric forcing in CrocoDash/MOM6 — NCAR runbook

**Prepared and documented by Faezeh Maghsoodifar**  
*Developed during the NCAR CROCODILE/MOM6 Workshop, September 2026.*

> This runbook records a learning and development effort built on the work of the CROCODILE, CrocoDash, CESM, CDEPS, CMEPS, MOM6, and NCAR communities. The author gratefully acknowledges the workshop instructors and project developers whose software, examples, and guidance made this work possible.

This notebook records the installation and execution workflow used on **Derecho/Casper** on 30 September 2026. It creates a private ERA5-enabled CrocoDash checkout and a new MOM6 case. It does **not** modify NCAR shared model code or existing cases.

## Where to run each instruction

Every section is labeled **Derecho shell** or **Jupyter notebook**. Run shell commands only in an NCAR terminal. Run Python cells only in JupyterHub with the **Python [conda env:CrocoDash]** kernel. Activating the Jupyter kernel does not activate a Derecho shell; they are separate processes. Never paste a shell command into a Python cell, and never run `qsub -q tutorial` by itself.

## What is ERA5 in this workflow?

All eight atmospheric streams are ERA5: 10 m `u` and `v`, mean sea-level pressure, 2 m temperature, specific humidity derived from ERA5 dew point and pressure, precipitation, downward shortwave radiation, and downward longwave radiation. `DATM%JRA` remains only as the CDEPS interface expected by the ocean coupler. Ocean initial/boundary conditions remain GLORYS, tides remain TPXO, and bathymetry remains GEBCO.


### ERA5 inputs and MOM6/CDEPS fields

The ERA5 preprocessing and atmosphere configurator use the following source-to-interface mapping:

| ERA5 input | Purpose in MOM6 |
| ---------- | --------------- |
| `10u` | Eastward 10 m wind, `Sa_u` |
| `10v` | Northward 10 m wind, `Sa_v` |
| `msl` | Mean sea-level pressure, `Sa_pslv` |
| `2t` | 2 m air temperature, `Sa_tbot` |
| `2d` | Used with pressure to calculate specific humidity |
| `mtpr` | Mean total precipitation rate, `Faxa_prec` |
| `msdwswrf` | Downward shortwave radiation, `Faxa_swdn` |
| `msdwlwrf` | Downward longwave radiation, `Faxa_lwdn` |

`2d` is not passed directly to the coupler. The preprocessing calculates specific humidity from ERA5 dew-point temperature and pressure and supplies the result as `Sa_shum`. The two wind components and the two radiation components make this a total of eight atmospheric streams.


## Locations and names

```text
Workspace:          /glade/work/faezehmaghso/crocodile2026
Original CrocoDash: /glade/work/faezehmaghso/crocodile2026/CrocoDash
Private ERA5 code:  /glade/work/faezehmaghso/crocodile2026/CrocoDash-era5
Conda environment:  /glade/work/faezehmaghso/conda-envs/CrocoDash
Handoff directory:  /glade/work/faezehmaghso/crocodile2026/era5_crocodash
ERA5 output:        /glade/work/faezehmaghso/crocodile2026/era5_forcing/ike2008
Local branch:       era5-atmosphere
Git commit:         5b45e986ac620c9138c6eab2a3a739957fe056d6
PBS queue:          tutorial
Project:            UCGD0009
```


## 1. Upload the package

The Windows package was created at:

```text
C:\Users\fmagh\OneDrive - The University of Alabama\Research\SummerSchool_Programs\NCARMOM6\era5_crocodash_ncar.zip
```

Upload it to `/glade/work/faezehmaghso/crocodile2026/`, then in a Derecho shell run:

```bash
cd /glade/work/faezehmaghso/crocodile2026
unzip era5_crocodash_ncar.zip
cd era5_crocodash
```


## 2. Read-only NCAR probe

Run in the **Derecho shell**:

```bash
cd /glade/work/faezehmaghso/crocodile2026/era5_crocodash
bash probe_ncar.sh | tee probe_ncar.out
```

The probe confirmed that the August and September 2008 RDA directories contain all required ERA5 products and that CESM contains the eight JRA stream definitions. The initial `ModuleNotFoundError` happened only because the CrocoDash conda environment was not active in that shell.


## 3. Verify the existing private CrocoDash environment

Run in the **Derecho shell**:

```bash
/glade/work/faezehmaghso/conda-envs/CrocoDash/bin/python \
  -c "import CrocoDash; print(CrocoDash.__file__)"

find /glade/work/faezehmaghso/conda-envs/CrocoDash \
  -type f -name 'ESMF_Scrip2Unstruct' -print
```

Observed results:

```text
/glade/work/faezehmaghso/crocodile2026/CrocoDash/CrocoDash/__init__.py
/glade/work/faezehmaghso/conda-envs/CrocoDash/bin/ESMF_Scrip2Unstruct
```


## 4. Verify the corrected portable Git bundle

The first bundle was generated from shallow Git history and was replaced. The corrected bundle must have this checksum:

```bash
sha256sum /glade/work/faezehmaghso/crocodile2026/era5_crocodash/crocodash-era5.bundle
```

```text
059bc9a7dedcb9c52873bc62d5b9168dcdb7afd5f6e152f452e264968ca8dbc4
```


## 5. Create the separate ERA5-enabled CrocoDash checkout

Run once in the **Derecho shell**:

```bash
cd /glade/work/faezehmaghso/crocodile2026

git clone --no-checkout \
  era5_crocodash/crocodash-era5.bundle \
  CrocoDash-era5

cd CrocoDash-era5
git checkout -b era5-atmosphere 5b45e986ac620c9138c6eab2a3a739957fe056d6
git submodule update --init --recursive
```

This populates the private checkout with its pinned `regional-mom6`, `visualCaseGen`, `mom6_forge`, and related submodules.


## 6. Connect the private conda environment to the ERA5 checkout

Run once in the **Derecho shell**:

```bash
/glade/work/faezehmaghso/conda-envs/CrocoDash/bin/python \
  -m pip install --no-deps -e \
  /glade/work/faezehmaghso/crocodile2026/CrocoDash-era5
```

Verify:

```bash
/glade/work/faezehmaghso/conda-envs/CrocoDash/bin/python -c \
"import inspect; from CrocoDash.forcing.atm import ERA5AtmosphereConfigurator; print(inspect.getfile(ERA5AtmosphereConfigurator)); print(ERA5AtmosphereConfigurator.name)"
```

Expected:

```text
/glade/work/faezehmaghso/crocodile2026/CrocoDash-era5/CrocoDash/forcing/atm.py
ERA5Atmosphere
```


## 7. Verify the Jupyter kernel

Open this workflow using the **Python [conda env:CrocoDash]** kernel. Restart the kernel after the editable installation, then run the next cell.


In [ ]:
import inspect
from CrocoDash.forcing.atm import ERA5AtmosphereConfigurator

print(inspect.getfile(ERA5AtmosphereConfigurator))
print(ERA5AtmosphereConfigurator.name)

assert ERA5AtmosphereConfigurator.name == "ERA5Atmosphere"
assert "CrocoDash-era5" in inspect.getfile(ERA5AtmosphereConfigurator)


## 8. Activate the environment in the Derecho shell

A Jupyter kernel and a terminal are separate processes. Activating the notebook kernel does not activate the Derecho shell. Before submitting the preprocessing job, run:

```bash
module load conda
conda activate /glade/work/faezehmaghso/conda-envs/CrocoDash
which python
which ESMF_Scrip2Unstruct
```

Both paths should start with `/glade/work/faezehmaghso/conda-envs/CrocoDash/bin/`.


## 9. Prepare the complete ERA5 atmosphere on the tutorial queue

Run in the activated **Derecho shell**:

```bash
cd /glade/work/faezehmaghso/crocodile2026/era5_crocodash
qsub -V prepare_era5_tutorial.pbs
```

Submission on 30 September 2026 returned job `7659803.desched1`. This is historical information—do not submit a duplicate merely to reproduce that job ID.

Verify the queue and status:

```bash
qstat 7659803.desched1
# After it finishes, use historical status:
qstat -x -f 7659803.desched1 | grep -E "job_state =|Exit_status =|queue ="
```

Job state `Q` means queued, `R` means running, and `F` means finished. The completed job reported `Exit_status = 0`. It ran in internal reservation queue `S7606485`; this is the execution queue to which the workshop `tutorial` queue routed the job.


## 10. Inspect the preprocessing result after the job finishes

Run in the **Derecho shell** only after the job is no longer queued or running:

```bash
cd /glade/work/faezehmaghso/crocodile2026/era5_crocodash
ls -lh era5_atmos_ike.o7659803*
tail -n 80 era5_atmos_ike.o7659803*

ls -lh /glade/work/faezehmaghso/crocodile2026/era5_forcing/ike2008/
```

Successful products observed on Derecho:

```text
era5_atmos_ike2008.nc    7.2 GB
era5_025_scrip.nc         84 MB
era5_025_ESMFmesh.nc      56 MB
```

The xarray chunk-alignment messages in the log are performance warnings, not data errors. The job printed all eight prepared fields and finished with exit status zero.


## 11. Create the new MOM6 case from the full ERA5 notebook

Only after the three forcing products exist, open:

```text
/glade/work/faezehmaghso/crocodile2026/era5_crocodash/ike_era5_full_crocodash.ipynb
```

Use the **CrocoDash** kernel and restart it. Run the cells in this exact **Jupyter notebook** order:

1. **Settings:** confirmed `ERA5AtmosphereConfigurator` came from `CrocoDash-era5`.
2. **Horizontal grid:** produced `nx=192`, `ny=156`, 29,952 cells.
3. **Topography:** xESMF bilinear regridding completed; maximum depth was 5,745 m.
4. **Open edges:** south had 71 wet cells, east had 80, north/west had zero; boundaries were `['south', 'east']`.
5. **Vertical grid:** top-layer thickness was about 4.917 m.
6. **Safety check**, followed by the `Case(...)` cell exactly once. The new case is `gom12_ike_ERA5full.001`. **Never rerun `Case(..., override=True)` after this point.**
7. **`case.configure_forcings(...)`:** activated `ERA5Atmosphere` and assigned all eight ERA5 fields. The JRA defaults printed first are overwritten by the ERA5 entries printed afterward.
8. **`case.process_forcings()`:** completed GLORYS initial/boundary conditions and TPXO tides in about 368 seconds.
9. **Extra MOM6 parameters:** added the remaining seven tidal constituents and skipped duplicate `DT`.
10. **SFINCS diagnostic table:** wrote six required output lines.

At that point, stop using Jupyter for setup and continue with the Derecho-shell verification below.


## 12. Verify atmospheric-forcing provenance

After the notebook creates and configures the case, run in the **Derecho shell**:

```bash
cd /glade/work/faezehmaghso/crocodile2026/croc_cases/gom12_ike_ERA5full.001
./case.setup --reset
./preview_namelists

RUNDIR=$(./xmlquery --value RUNDIR)
grep -c "era5_atmos_ike2008.nc" "$RUNDIR/datm.streams.xml"
grep -c "JRA.v1.5" "$RUNDIR/datm.streams.xml"
grep -n "era5_atmos_ike2008.nc" "$RUNDIR/datm.streams.xml"
```

Required result:

```text
ERA5 references: 8
JRA data-file references: 0
```

The eight ERA5 references correspond to precipitation, downward longwave radiation, downward shortwave radiation, 2 m specific humidity, mean sea-level pressure, 2 m temperature, and both 10 m wind components. `DATM%JRA` remains the CDEPS field interface name, but the generated stream configuration reads no JRA forcing file.


## 13. Confirm tutorial queue, build, and submit the MOM6 case

Continue in the new case directory in the **Derecho shell**:

```bash
./xmlchange JOB_QUEUE=tutorial --force
./xmlquery JOB_QUEUE,JOB_WALLCLOCK_TIME
./preview_run

# Build only after preview_run shows 'qsub -q tutorial' for case.run:
qcmd -A UCGD0009 -- ./case.build

# Submit only after the build ends with MODEL BUILD HAS FINISHED SUCCESSFULLY:
./case.submit
qstat -u faezehmaghso
```

Observed `preview_run` output showed `qsub -q tutorial` for `case.run`, `case.st_archive`, and `case.cupid`. The generated `.case.run` did not contain a `#PBS -q` line because CIME supplies the queue on the `qsub` command line. The `qcmd` build helper may report the workshop development queue; it is not the model's `case.run` queue.

**Do not run `qsub -q tutorial` manually.** If it prints `Job script will be read from standard input`, press **Ctrl+C**, not Ctrl+D, and then use the correct build command: `qcmd -A UCGD0009 -- ./case.build`. Later, use `./case.submit`; CIME constructs the full `qsub -q tutorial ...` command.

### Results from this case

The build completed in about 534 seconds and ended with `MODEL BUILD HAS FINISHED SUCCESSFULLY`. CIME then submitted:

```text
case.run:        7661113.desched1
case.st_archive: 7661114.desched1 (depends on successful case.run)
```

Both submission commands used `qsub -q tutorial`. While running, PBS showed internal reservation queue `S7606485`, which is the execution queue reached through the workshop `tutorial` queue.

Monitor the job in the **Derecho shell**:

```bash
qstat -f 7661113.desched1 | grep -E "job_state =|queue =|resources_used.walltime"

RUNDIR=$(./xmlquery --value RUNDIR)
LOG=$(ls -1t "$RUNDIR"/cesm.log.* | head -1)
echo "$LOG"
tail -n 60 "$LOG"
```

### Email notifications

For a future submission, request completion/failure email through CIME:

```bash
./case.submit --mail-user YOUR_EMAIL -M end -M fail
```

For an already submitted PBS job, do not resubmit. Add abort/end notification with `qalter`:

```bash
qalter -M YOUR_EMAIL -m ae JOB_ID
qstat -f JOB_ID | grep -E "Mail_Users|Mail_Points"
```

For the repaired run, notifications were confirmed on model job `7661567.desched1` and archive job `7661568.desched1` with `Mail_Points = ae`. An email about `7661114.desched1` being deleted referred to the obsolete archive dependency from the first failed run and was expected.


## 14. Repair for `deptho AREA measures field NOT found`

The first case attempt reached MOM6 initialization but stopped because the minimal SFINCS `diag_table` listed `deptho` without its required AREA-measure field, `areacello`. This is not an ERA5 failure and does not require rebuilding the executable. The corrected notebooks now add `areacello` automatically.

For a case created by an older notebook, add the following line to `SourceMods/src.mom/diag_table`, replacing `CASE_NAME` with the actual case name:

```text
"ocean_model", "areacello", "areacello", "CASE_NAME.mom6.h.static", "all", "none", "none", 2
```

Then run in the **Derecho shell**:

```bash
./preview_namelists
RUNDIR=$(./xmlquery --value RUNDIR)
for f in SourceMods/src.mom/diag_table CaseDocs/diag_table "$RUNDIR/diag_table"; do
  echo "===== $f"
  grep -nE '"(areacello|deptho)"' "$f"
done
```

Both fields must appear in all three files. Cancel only the obsolete archive job associated with the failed attempt, then run `./case.submit`. Do not rebuild and do not run `qsub` manually.

## 15. Safe rollback to the original CrocoDash checkout

If the private environment should later point back to the original workshop checkout, run:

```bash
/glade/work/faezehmaghso/conda-envs/CrocoDash/bin/python \
  -m pip install --no-deps -e \
  /glade/work/faezehmaghso/crocodile2026/CrocoDash
```

This rollback changes only the editable Python-package link in the user's private conda environment. It does not delete either checkout or any model case.

### Recorded failure and recovery

The original model job `7661113.desched1` failed during diagnostic initialization. Its dependent archive job `7661114.desched1` was deleted automatically. After adding `areacello`, `./preview_namelists` regenerated the runtime table and the existing executable was reused. No rebuild was required.

The repaired jobs completed successfully:

```text
model:   7661567.desched1   Exit_status=0   walltime=01:02:10
archive: 7661568.desched1   Exit_status=0   walltime=00:00:06
```

Use these commands to verify the historical records:

```bash
qstat -x -f 7661567.desched1 | grep -E "job_state =|Exit_status =|resources_used.walltime"
qstat -x -f 7661568.desched1 | grep -E "job_state =|Exit_status =|resources_used.walltime"
tail -n 20 CaseStatus
```


## Current checkpoint

- Required ERA5 files: confirmed on NCAR RDA.
- Eight CDEPS JRA interface streams: confirmed.
- Private ERA5 CrocoDash checkout and submodules: installed.
- Derecho Python import: confirmed from `CrocoDash-era5`.
- Jupyter kernel import: confirmed from `CrocoDash-era5`.
- ESMF mesh converter: confirmed.
- ERA5 preparation job `7659803.desched1`: finished successfully with exit status zero.
- ERA5 forcing NetCDF, SCRIP grid, and ESMF mesh: created successfully.
- Horizontal grid, topography, vertical grid, open boundaries, GLORYS conditions, and TPXO tides: created successfully.
- New case: `/glade/work/faezehmaghso/crocodile2026/croc_cases/gom12_ike_ERA5full.001`.
- Final `datm.streams.xml`: verified exactly eight references to `era5_atmos_ike2008.nc` and zero references to `JRA.v1.5`.
- `preview_run`: verified `qsub -q tutorial` for all case jobs.
- Build: completed successfully in about 534 seconds.
- First run attempt `7661113.desched1`: stopped during MOM6 diagnostic initialization.
- Root cause: the minimal SFINCS `diag_table` included `deptho` but omitted its required `areacello` AREA-measure field. This was not an ERA5, CDEPS-stream, compiler, or mesh failure.
- Obsolete archive job `7661114.desched1`: automatically deleted because its parent failed, as expected.
- The full ERA5 notebook was corrected to always include `areacello` and to repair an existing minimal CaseDocs table that omits it.
- Repaired model run `7661567.desched1`: completed successfully with exit status zero in 01:02:10.
- Repaired archive job `7661568.desched1`: completed successfully with exit status zero in 00:00:06.
- `CaseStatus`: model execution, case.run, timing generation, and short-term archive all recorded success.
- Final restart date: `2008-09-20-00000`, confirming completion of the 21-day simulation.
- Hourly SSH: 504 records on the 192 x 156 grid, exactly 21 days.
- Texas SFINCS region: joined 12 FMS tiles into a 20.7 MB file on a 55 x 31 grid, covering 2008-08-30 00:10 through 2008-09-20 00:00.
- Mississippi SFINCS region: joined 12 FMS tiles into a 15.1 MB file on a 43 x 29 grid over the same period.
- Permanent analysis directory: `/glade/work/faezehmaghso/crocodile2026/mom6_output/gom12_ike_ERA5full.001/`.
- Final status: full ERA5 atmospheric forcing integration and MOM6 case completed successfully.


## 16. Final validation and presentation screenshot

Run this block in the **Derecho shell** after the case finishes. It recreates the concise evidence used in the presentation without depending on variables from an earlier terminal session.

```bash
CASE=/glade/work/faezehmaghso/crocodile2026/croc_cases/gom12_ike_ERA5full.001
cd "$CASE"

RUNDIR=$(./xmlquery --value RUNDIR)
ERA5_COUNT=$(grep -c "era5_atmos_ike2008.nc" "$RUNDIR/datm.streams.xml")
JRA_COUNT=$(grep -c "JRA.v1.5" "$RUNDIR/datm.streams.xml")

printf "\nMOM6 ATMOSPHERIC FORCING VALIDATION\n"
printf "Case: %s\n\n" "$(basename "$CASE")"
printf "ERA5 stream references : %s of 8\n" "$ERA5_COUNT"
printf "JRA file references    : %s\n\n" "$JRA_COUNT"
printf "Atmospheric source:\n"
grep -m 1 "era5_atmos_ike2008.nc" "$RUNDIR/datm.streams.xml"
printf "\nRun status:\n"
grep -E "model execution success|st_archive success" CaseStatus | tail -2
```

The final result must report **8 of 8 ERA5 streams**, **0 JRA forcing-file references**, and successful model and archive status. This verifies forcing provenance and run completion together.

### Final output checks

```bash
DOUT=$(./xmlquery --value DOUT_S_ROOT)
RUNDIR=$(./xmlquery --value RUNDIR)

find "$DOUT" -maxdepth 4 -type f -name "*.nc" | sort
find "$RUNDIR" -maxdepth 1 -type f -name "*.nc" -printf "%10s  %f\n" | sort -k2
find "$RUNDIR" -maxdepth 1 -type f -iname "*sfincs*" -print
```

The post-processing cell in `ike_era5_full_crocodash.ipynb` joins the 12 Texas tiles and 12 Mississippi tiles, copies hourly SSH and daily surface output from scratch, and preserves them in:

```text
/glade/work/faezehmaghso/crocodile2026/mom6_output/gom12_ike_ERA5full.001/
```
